In [15]:
!pip install backports.zoneinfo

In [49]:
import pandas as pd

df = pd.read_parquet("/Users/rk8896/Downloads/mag7_news_with_sentiment_and_topics_labeledV2.parquet 2")

df['year'] = pd.to_datetime(df['published_at']).dt.year

df.head()

df.count()

article_id           101346
published_at         101346
title                101346
content              101346
url                  101346
source               101346
symbols              101346
tags                 101346
sentiment_vendor     101329
symbol_query         101346
fetch_date           101346
chrono_model         101346
text_for_nlp         101346
topic_id_kmeans      101346
sent_neg             101346
sent_neu             101346
sent_pos             101346
sentiment_finbert    101346
topic_label_auto     101346
year                 101346
dtype: int64

In [50]:
pd.set_option('display.max_rows', None)

In [51]:
schema = {
    'topic_id': 'Int32',    # Use nullable integer
    'label': 'string',  # Use modern string type
}

map_df = pd.read_csv('/Users/rk8896/DSO585/stock-news-autoencoder/topic_to_label_map_v2.csv', encoding="utf-8", 
                     dtype = schema)

map_df.info()

map_df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   topic_id  50 non-null     Int32 
 1   label     50 non-null     string
dtypes: Int32(1), string(1)
memory usage: 778.0 bytes


,topic_id,label
0,0,Investment Analysis & Strategy
1,1,Corporate Strategy & Growth
2,2,Investment Analysis & Strategy
3,3,CEO & Influencer News
4,4,Geopolitical & Trade News
5,5,CEO & Influencer News
6,6,Semiconductor & Chip Industry News
7,7,Market Movements & Trading News
8,8,Legal & Regulatory Changes
9,9,Legal & Regulatory Changes


In [54]:
import numpy as np

df1 = pd.merge(df, map_df, how='left',
                     left_on='topic_id_kmeans', right_on='topic_id')

# ==========================
# 1️⃣ Manual NASDAQ holiday list (can extend for future years)
# ==========================
nasdaq_holidays = ['2023-01-02', '2023-01-16', '2023-02-20', '2023-04-07', '2023-05-29', 
 '2023-06-19', '2023-07-04', '2023-09-04', '2023-11-23', '2023-12-25',
 '2024-01-01', '2024-01-15', '2024-02-19', '2024-03-29', '2024-05-27', 
 '2024-06-19', '2024-07-04', '2024-09-02', '2024-11-28', '2024-12-25',
 '2025-01-01', '2025-01-20', '2025-02-17', '2025-04-18', '2025-05-26', 
 '2025-06-19', '2025-07-04', '2025-09-01', '2025-11-27', '2025-12-25']
nasdaq_holidays = pd.to_datetime(nasdaq_holidays).date
holidays_np = np.array(nasdaq_holidays, dtype='datetime64[D]')

# ==========================
# 2️⃣ Vectorized adjustment
# ==========================
cutoff_hour = 16

# Step 1: move timestamps after cutoff to next day
next_days = df1['published_at'].copy()
next_days += pd.to_timedelta((next_days.dt.hour >= cutoff_hour).astype(int), unit='D')

# Step 2: normalize time to midnight
next_days = next_days.dt.normalize()

# Step 3: vectorized loop to skip weekends and holidays
mask = (next_days.dt.weekday >= 5) | np.isin(next_days.dt.date, holidays_np)
while mask.any():
    next_days[mask] += pd.Timedelta(days=1)
    mask = (next_days.dt.weekday >= 5) | np.isin(next_days.dt.date, holidays_np)

# Step 4: assign back
df1['final_date_for_news'] = next_days

df1['final_date_for_news'] = df1['final_date_for_news'].dt.strftime('%Y-%m-%d')

#df1[df1['published_at'].between('2023-01-13', '2023-01-16')]

df1.head(20)

df1.to_parquet('/Users/rk8896/DSO585/stock-news-autoencoder/news_raw_withfinal_topics.parquet')

<ipython-input-54-419e81e13dee>:33: SettingWithCopyWarning: modifications to a method of a datetimelike object are not supported and are discarded. Change values on the original.
  next_days[mask] += pd.Timedelta(days=1)


In [34]:
grouped_df = df1.groupby(['symbol_query', 'final_date_for_news', 'label']).agg(
    sentiment_neg=('sent_neg', 'mean'),      # Collect list of users
    sentiment_neutral=('sent_neu', 'mean'), # Collect unique users
    sentiment_pos=('sent_pos', 'mean'),    # Collect list of ratings
    sentiment_finbert=('sentiment_finbert', 'mean'),     # Get average rating
    total_count=('symbol_query', 'size'),      # <-- ADDED: Count of all rows in group
).reset_index()

print(grouped_df['sentiment_neg'].min())
print(grouped_df['sentiment_neutral'].min())
print(grouped_df['sentiment_pos'].min())
print(grouped_df['sentiment_finbert'].min())

grouped_df.head(30)

0.0062589929439127445
0.006491740699857473
0.008695622906088829
-0.9404048919677734


,symbol_query,final_date_for_news,label,sentiment_neg,sentiment_neutral,sentiment_pos,sentiment_finbert,total_count
0,AAPL.US,2023-01-03,CEO & Influencer News,0.362340,0.173303,0.464357,0.102018,6
1,AAPL.US,2023-01-03,Corporate Earnings & Financials,0.008590,0.975990,0.015421,0.006831,1
2,AAPL.US,2023-01-03,Corporate Strategy & Growth,0.010106,0.968566,0.021328,0.011222,1
3,AAPL.US,2023-01-03,Economic Indicators & Fed Policy,0.010160,0.974552,0.015288,0.005128,1
4,AAPL.US,2023-01-03,Geopolitical & Trade News,0.127329,0.516956,0.355716,0.228387,6
5,AAPL.US,2023-01-03,Innovation & Future Tech,0.226984,0.324224,0.448792,0.221808,4
6,AAPL.US,2023-01-03,Investment & Hedge Fund Activity,0.274730,0.070090,0.655180,0.380450,4
7,AAPL.US,2023-01-03,Investment Analysis & Strategy,0.017554,0.708444,0.274002,0.256449,1
8,AAPL.US,2023-01-03,Layoffs & Corporate Restructuring,0.600182,0.020008,0.379811,-0.220371,1
9,AAPL.US,2023-01-03,Market Movements & Trading News,0.054555,0.474737,0.470708,0.416152,2


In [35]:
pivoted_df = grouped_df.pivot(
    index=['symbol_query', 'final_date_for_news'],  # Columns to keep as rows
    columns='label',                         # Column to pivot into new columns
    values=[                                 # Values to fill the new columns
        'sentiment_finbert',
        'total_count'
    ]
).fillna(0)

# Flatten MultiIndex columns
pivoted_df.columns = [f"{val}_{label}" for val, label in pivoted_df.columns]

pivoted_df = pivoted_df.reset_index()

pivoted_df.head(30)

,symbol_query,final_date_for_news,sentiment_finbert_CEO & Influencer News,sentiment_finbert_Corporate Earnings & Financials,sentiment_finbert_Corporate Strategy & Growth,sentiment_finbert_Economic Indicators & Fed Policy,sentiment_finbert_General Business & News Reporting,sentiment_finbert_Geopolitical & Trade News,sentiment_finbert_Innovation & Future Tech,sentiment_finbert_Investment & Hedge Fund Activity,...,total_count_Geopolitical & Trade News,total_count_Innovation & Future Tech,total_count_Investment & Hedge Fund Activity,total_count_Investment Analysis & Strategy,total_count_Layoffs & Corporate Restructuring,total_count_Legal & Regulatory Changes,total_count_Market Movements & Trading News,"total_count_Mergers, Acquisitions & Deals",total_count_Product News & Updates,total_count_Semiconductor & Chip Industry News
0,AAPL.US,2023-01-03,0.102018,0.006831,0.011222,0.005128,0.000000,0.228387,0.221808,0.380450,...,6.0,4.0,4.0,1.0,1.0,0.0,2.0,0.0,2.0,0.0
1,AAPL.US,2023-01-04,0.295232,0.004108,0.105869,-0.028463,0.187918,-0.254930,0.246283,0.000000,...,3.0,3.0,0.0,1.0,3.0,0.0,10.0,0.0,1.0,0.0
2,AAPL.US,2023-01-05,-0.003781,-0.020834,0.032395,0.011563,0.000000,0.050943,-0.000648,-0.484566,...,3.0,1.0,1.0,0.0,0.0,1.0,3.0,0.0,10.0,0.0
3,AAPL.US,2023-01-06,0.339910,0.006405,-0.081714,-0.328843,0.000000,-0.174730,0.095464,-0.058425,...,1.0,3.0,2.0,0.0,2.0,0.0,3.0,0.0,1.0,0.0
4,AAPL.US,2023-01-09,0.402631,0.000000,0.592804,-0.002143,0.000000,0.123715,0.017661,0.000000,...,9.0,1.0,0.0,0.0,0.0,0.0,4.0,0.0,3.0,0.0
5,AAPL.US,2023-01-10,0.235297,-0.087694,0.171762,-0.230516,0.000000,0.009193,-0.154745,0.000000,...,5.0,5.0,0.0,0.0,6.0,0.0,4.0,0.0,11.0,0.0
6,AAPL.US,2023-01-11,0.233529,0.005111,0.000000,-0.397814,-0.341970,0.384314,-0.072696,-0.915715,...,1.0,4.0,1.0,0.0,3.0,3.0,4.0,0.0,9.0,0.0
7,AAPL.US,2023-01-12,0.557647,-0.939582,0.890311,0.000000,0.435846,0.106334,0.129372,0.000000,...,5.0,4.0,0.0,0.0,3.0,0.0,7.0,0.0,2.0,0.0
8,AAPL.US,2023-01-13,0.513775,0.027287,0.000000,0.000000,0.765312,-0.028932,0.000000,-0.306461,...,5.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,3.0,0.0
9,AAPL.US,2023-01-17,0.345622,0.000000,-0.050851,-0.012620,0.154174,0.147139,0.649367,0.194337,...,4.0,6.0,5.0,1.0,6.0,0.0,8.0,0.0,10.0,0.0


In [66]:
pivoted_df.to_parquet('/Users/rk8896/DSO585/stock-news-autoencoder/aggregated_results_topics_sentiment.parquet')

In [45]:
df1.to_parquet('/Users/rk8896/DSO585/stock-news-autoencoder/final_df_with_label_v2.parquet')

In [40]:
yf_df = pd.read_parquet('/Users/rk8896/DSO585/stock-news-autoencoder/mag7_yf_2021_2025.parquet')

yf_df.tail(20)

,symbol_query,date,day_start_value,day_max_value,day_min_value,day_end_value,day_end_raw_close,ret_1d,ret_log_1d,ret_5d,...,market_day_end_value,market_day_end_raw_close,market_ret_1d,market_ret_log_1d,market_ret_5d,market_ret_log_5d,market_ret_10d,market_ret_log_10d,market_ret_21d,market_ret_log_21d
8471,TSLA.US,2025-10-03,429.829987,429.829987,446.769989,443.290009,416.579987,-0.043598,-0.044577,-0.010546,...,669.210022,669.210022,-0.000015,-0.000015,0.011166,0.011104,0.008302,0.008268,0.030950,0.030480
8472,TSLA.US,2025-10-06,453.250000,453.250000,453.549988,440.750000,436.690002,0.048274,0.047145,-0.006394,...,671.609985,671.609985,0.003586,0.003580,0.011949,0.011878,0.007153,0.007128,0.037652,0.036961
8473,TSLA.US,2025-10-07,433.089996,433.089996,452.679993,447.820007,432.450012,-0.009709,-0.009757,-0.001547,...,669.119995,669.119995,-0.003707,-0.003714,0.004413,0.004404,0.008911,0.008872,0.031272,0.030793
8474,TSLA.US,2025-10-08,438.690002,438.690002,441.329987,437.570007,425.230011,-0.016696,-0.016837,-0.035213,...,673.109985,673.109985,0.005963,0.005945,0.006971,0.006947,0.018167,0.018004,0.035028,0.034429
8475,TSLA.US,2025-10-09,435.540009,435.540009,436.350006,431.809998,426.179993,0.002234,0.002232,-0.021558,...,671.159973,671.159973,-0.002897,-0.002901,0.002899,0.002895,0.019922,0.019727,0.029055,0.028641
8476,TSLA.US,2025-10-10,413.489990,413.489990,443.130005,436.540009,411.450012,-0.034563,-0.035174,-0.012315,...,653.020020,653.020020,-0.027028,-0.027400,-0.024193,-0.024490,-0.013297,-0.013386,-0.007010,-0.007035
8477,TSLA.US,2025-10-13,435.899994,435.899994,436.890015,423.529999,419.700012,0.020051,0.019853,-0.038906,...,663.039978,663.039978,0.015344,0.015228,-0.012760,-0.012843,-0.000964,-0.000965,0.008564,0.008527
8478,TSLA.US,2025-10-14,429.239990,429.239990,434.200012,426.790009,417.859985,-0.004384,-0.004394,-0.033738,...,662.229980,662.229980,-0.001222,-0.001222,-0.010297,-0.010351,-0.005929,-0.005947,0.001997,0.001995
8479,TSLA.US,2025-10-15,435.149994,435.149994,440.510010,434.899994,426.329987,0.020270,0.020067,0.002587,...,665.169983,665.169983,0.004440,0.004430,-0.011796,-0.011866,-0.004907,-0.004919,0.007833,0.007803
8480,TSLA.US,2025-10-16,428.750000,428.750000,439.350006,434.730011,421.309998,-0.011775,-0.011845,-0.011427,...,660.640015,660.640015,-0.006810,-0.006834,-0.015674,-0.015798,-0.012821,-0.012904,0.002215,0.002212


In [41]:
yf_df['date'] = pd.to_datetime(yf_df['date'])

pivoted_df['final_date_for_news'] = pd.to_datetime(pivoted_df['final_date_for_news'])

yf_df.info()

pivoted_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8491 entries, 0 to 8490
Data columns (total 28 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   symbol_query              8491 non-null   object        
 1   date                      8491 non-null   datetime64[ns]
 2   day_start_value           8491 non-null   float64       
 3   day_max_value             8491 non-null   float64       
 4   day_min_value             8491 non-null   float64       
 5   day_end_value             8491 non-null   float64       
 6   day_end_raw_close         8491 non-null   float64       
 7   ret_1d                    8484 non-null   float64       
 8   ret_log_1d                8484 non-null   float64       
 9   ret_5d                    8456 non-null   float64       
 10  ret_log_5d                8456 non-null   float64       
 11  ret_10d                   8421 non-null   float64       
 12  ret_log_10d         

In [42]:
final_df = pd.merge(pivoted_df, yf_df, how='left',
                     left_on=['symbol_query', 'final_date_for_news'], right_on=['symbol_query', 'date'])

final_df.head(30)

,symbol_query,final_date_for_news,sentiment_finbert_CEO & Influencer News,sentiment_finbert_Corporate Earnings & Financials,sentiment_finbert_Corporate Strategy & Growth,sentiment_finbert_Economic Indicators & Fed Policy,sentiment_finbert_General Business & News Reporting,sentiment_finbert_Geopolitical & Trade News,sentiment_finbert_Innovation & Future Tech,sentiment_finbert_Investment & Hedge Fund Activity,...,market_day_end_value,market_day_end_raw_close,market_ret_1d,market_ret_log_1d,market_ret_5d,market_ret_log_5d,market_ret_10d,market_ret_log_10d,market_ret_21d,market_ret_log_21d
0,AAPL.US,2023-01-03,0.102018,0.006831,0.011222,0.005128,0.000000,0.228387,0.221808,0.380450,...,367.150726,380.820007,-0.004210,-0.004219,-0.005458,-0.005473,-0.006392,-0.006413,-0.065197,-0.067420
1,AAPL.US,2023-01-04,0.295232,0.004108,0.105869,-0.028463,0.187918,-0.254930,0.246283,0.000000,...,369.985291,383.760010,0.007720,0.007691,0.006188,0.006169,0.009842,0.009794,-0.056892,-0.058575
2,AAPL.US,2023-01-05,-0.003781,-0.020834,0.032395,0.011563,0.000000,0.050943,-0.000648,-0.484566,...,365.762451,379.380005,-0.011413,-0.011479,0.007221,0.007195,-0.003048,-0.003053,-0.050577,-0.051901
3,AAPL.US,2023-01-06,0.339910,0.006405,-0.081714,-0.328843,0.000000,-0.174730,0.095464,-0.058425,...,374.150116,388.079987,0.022932,0.022673,0.012101,0.012028,0.004790,0.004778,-0.014600,-0.014708
4,AAPL.US,2023-01-09,0.402631,0.000000,0.592804,-0.002143,0.000000,0.123715,0.017661,0.000000,...,373.938049,387.859985,-0.000567,-0.000567,0.014199,0.014099,0.018754,0.018580,-0.013481,-0.013572
5,AAPL.US,2023-01-10,0.235297,-0.087694,0.171762,-0.230516,0.000000,0.009193,-0.154745,0.000000,...,376.560394,390.579987,0.007013,0.006988,0.025629,0.025306,0.020031,0.019833,-0.014284,-0.014387
6,AAPL.US,2023-01-11,0.233529,0.005111,0.000000,-0.397814,-0.341970,0.384314,-0.072696,-0.915715,...,381.323120,395.519989,0.012648,0.012569,0.030644,0.030184,0.037021,0.036353,0.005696,0.005680
7,AAPL.US,2023-01-12,0.557647,-0.939582,0.890311,0.000000,0.435846,0.106334,0.129372,0.000000,...,382.711426,396.959991,0.003641,0.003634,0.046339,0.045297,0.053895,0.052493,-0.004988,-0.005001
8,AAPL.US,2023-01-13,0.513775,0.027287,0.000000,0.000000,0.765312,-0.028932,0.000000,-0.306461,...,384.196106,398.500000,0.003880,0.003872,0.026850,0.026496,0.039276,0.038524,-0.008632,-0.008670
9,AAPL.US,2023-01-17,0.345622,0.000000,-0.050851,-0.012620,0.154174,0.147139,0.649367,0.194337,...,383.492371,397.769989,-0.001832,-0.001834,0.025550,0.025230,0.040112,0.039328,-0.004081,-0.004089


In [43]:
final_df.to_parquet('/Users/rk8896/DSO585/stock-news-autoencoder/aggregated_results_topics_sentiment_yf.parquet')

In [48]:
final_df12 = pd.read_parquet('/Users/rk8896/DSO585/stock-news-autoencoder/aggregated_results_topics_sentiment_yf.parquet')

final_df12.count()

symbol_query                                            4604
final_date_for_news                                     4604
sentiment_finbert_CEO & Influencer News                 4604
sentiment_finbert_Corporate Earnings & Financials       4604
sentiment_finbert_Corporate Strategy & Growth           4604
sentiment_finbert_Economic Indicators & Fed Policy      4604
sentiment_finbert_General Business & News Reporting     4604
sentiment_finbert_Geopolitical & Trade News             4604
sentiment_finbert_Innovation & Future Tech              4604
sentiment_finbert_Investment & Hedge Fund Activity      4604
sentiment_finbert_Investment Analysis & Strategy        4604
sentiment_finbert_Layoffs & Corporate Restructuring     4604
sentiment_finbert_Legal & Regulatory Changes            4604
sentiment_finbert_Market Movements & Trading News       4604
sentiment_finbert_Mergers, Acquisitions & Deals         4604
sentiment_finbert_Product News & Updates                4604
sentiment_finbert_Semico

In [45]:
final_df12.isna().sum()

symbol_query                                             0
final_date_for_news                                      0
sentiment_finbert_CEO & Influencer News                  0
sentiment_finbert_Corporate Earnings & Financials        0
sentiment_finbert_Corporate Strategy & Growth            0
sentiment_finbert_Economic Indicators & Fed Policy       0
sentiment_finbert_General Business & News Reporting      0
sentiment_finbert_Geopolitical & Trade News              0
sentiment_finbert_Innovation & Future Tech               0
sentiment_finbert_Investment & Hedge Fund Activity       0
sentiment_finbert_Investment Analysis & Strategy         0
sentiment_finbert_Layoffs & Corporate Restructuring      0
sentiment_finbert_Legal & Regulatory Changes             0
sentiment_finbert_Market Movements & Trading News        0
sentiment_finbert_Mergers, Acquisitions & Deals          0
sentiment_finbert_Product News & Updates                 0
sentiment_finbert_Semiconductor & Chip Industry News    

In [47]:
final_df12[final_df12.market_day_start_value.isna()]

,symbol_query,final_date_for_news,sentiment_finbert_CEO & Influencer News,sentiment_finbert_Corporate Earnings & Financials,sentiment_finbert_Corporate Strategy & Growth,sentiment_finbert_Economic Indicators & Fed Policy,sentiment_finbert_General Business & News Reporting,sentiment_finbert_Geopolitical & Trade News,sentiment_finbert_Innovation & Future Tech,sentiment_finbert_Investment & Hedge Fund Activity,...,market_day_end_value,market_day_end_raw_close,market_ret_1d,market_ret_log_1d,market_ret_5d,market_ret_log_5d,market_ret_10d,market_ret_log_10d,market_ret_21d,market_ret_log_21d
500,AAPL.US,2025-01-09,0.832838,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
664,AAPL.US,2025-10-31,-0.028852,-0.548224,-0.267431,0.0,0.301464,-0.428045,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
665,AAPL.US,2025-11-03,0.000000,-0.288913,0.000000,0.0,0.289533,-0.818845,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1169,AMZN.US,2025-01-09,0.715310,0.000000,-0.497410,0.0,0.000000,0.000000,0.000000,-0.703061,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1343,AMZN.US,2025-10-31,0.408007,-0.238201,-0.052200,0.0,0.785614,-0.218983,-0.924664,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1344,AMZN.US,2025-11-03,0.000000,-0.333667,-0.838771,0.0,-0.003485,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1843,GOOGL.US,2025-01-09,0.000000,0.000000,-0.821948,0.0,0.000000,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016,GOOGL.US,2025-10-31,0.526015,-0.818434,0.696868,0.0,0.642644,-0.044133,-0.898344,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017,GOOGL.US,2025-11-03,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2486,META.US,2025-01-09,0.600437,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df.pivot_table(columns = 'year', index = 'symbol_query', aggfunc = 'size')

#df_groups = df.groupby(['symbol_query', 'year']).count()

#df_groups.to_csv("/Users/rk8896/Downloads/group_counts.csv")

year,2023,2024,2025
symbol_query,,,
AAPL.US,7182.0,4970.0,7646.0
ABBV.US,713.0,424.0,748.0
ABT.US,507.0,254.0,558.0
AMAT.US,386.0,243.0,470.0
AMD.US,729.0,908.0,2347.0
AMGN.US,460.0,262.0,446.0
AMZN.US,5307.0,3187.0,4467.0
AON.US,112.0,104.0,224.0
AVGO.US,637.0,761.0,1940.0


In [16]:
df1 = df[df['symbol_query'] == 'GOOGL.US']

df1['symbols'].value_counts()

#df1.pivot_table(columns = 'year', index = 'symbol_query', aggfunc = 'size')


symbols
[GOOGL.US]                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         523
[ABEA.F, ABEA.XETRA, ABEC.F, ABEC.XETRA, GOGL34.SA, GOGL35.SA, GOOG.MI, GOOG.MX, GOOG.US, GOOGL.US]                                                                                                                                                                                                                                                                      

df.pivot_table(columns = 'year', index = 'symbol_query', aggfunc = 'size')

#df_groups = df.groupby(['symbol_query', 'year']).count()

#df_groups.to_csv("/Users/rk8896/Downloads/group_counts.csv")